## Análisis Exploratorio de Datos (EDA)

### Objetivo
Este notebook realiza un **análisis exploratorio** sobre los datasets extraídos del INE para verificar la integridad, calidad y estructura de los datos antes de las transformaciones.

Se examinan los siete DataFrames generados en la fase de extracción:
- **Empresas constituidas** (`empresas_constituidas.csv`) — Verificación de tipos societarios, consistencia de capital y nº de sociedades.
- **Empresas disueltas** (`empresas_disueltas.csv`) — Distribución por causa de disolución y territorio.
- **IPC** (`ipc.csv`) — Rango de valores, detección de outliers y cardinalidad de dimensiones.
- **Tablas dimensionales** (`sectores_ipc`, `territorio`, `tiempo`, `tipo_medida`) — Validación de claves y completitud.

### Metodología
1. **Carga** de los CSV desde `../files/data_raw/`.
2. **Inspección** mediante `info()`, `describe()`, `sample()` y `value_counts()`.
3. **Detección de inconsistencias** — Identificación de filas duplicadas (como los agregados "Mercantiles" que sumarizan a S.A. y S.L.) y valores anómalos.
4. **Documentación de hallazgos** — Conclusiones que guiarán las transformaciones en la siguiente etapa del pipeline.

In [ ]:
# Importación de librerías
import pandas as pd

# Configuración para visualizar todas las columnas
pd.set_option('display.max_columns', None)

1. Revisamos los archivos exportados para comprobar la integridad de los datos

In [2]:
df_empr_const = pd.read_csv('../files/data_raw/empresas_constituidas.csv', index_col=0)

In [3]:
df_empr_const.sample(5)

,territorio,id_tiempo,tipo,numero_sociedades,capital
id_const,,,,,
9101,"Balears, Illes",202302,Sociedades de responsabilidad limitada,388,15752000
10730,Extremadura,201604,Sociedades de responsabilidad limitada,107,3302000
12082,"Rioja, La",201402,Sociedades de responsabilidad limitada,43,2989000
16638,Melilla,202103,S. Comanditarias y S. Colectivas,0,0
4818,"Asturias, Principado de",201109,Sociedades anónimas,0,0


In [4]:
df_empr_const.info()

<class 'pandas.DataFrame'>
RangeIndex: 16796 entries, 1 to 16796
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   territorio         16796 non-null  str  
 1   id_tiempo          16796 non-null  int64
 2   tipo               16796 non-null  str  
 3   numero_sociedades  16796 non-null  int64
 4   capital            16796 non-null  int64
dtypes: int64(3), str(2)
memory usage: 656.2 KB


In [5]:
df_empr_const.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_tiempo,16796.0,2.016779e+05,5.318978e+02,200801.0,201208.0,201703.0,202110.0,2.026050e+05
numero_sociedades,16796.0,2.137710e+02,4.479916e+02,0.0,0.0,7.0,216.0,3.173000e+03
capital,16796.0,1.544515e+07,1.486531e+08,0.0,0.0,444000.0,7887000.0,1.259172e+10


In [6]:
df_empr_const.describe(include='string').T

,count,unique,top,freq
territorio,16796,19,Andalucía,884
tipo,16796,4,Mercantiles,4199


In [7]:
df_empr_const["territorio"].unique()

<StringArray>
[                  'Andalucía',                      'Aragón',
     'Asturias, Principado de',              'Balears, Illes',
                    'Canarias',                   'Cantabria',
             'Castilla y León',        'Castilla - La Mancha',
                    'Cataluña',        'Comunitat Valenciana',
                 'Extremadura',                     'Galicia',
        'Madrid, Comunidad de',           'Murcia, Región de',
 'Navarra, Comunidad Foral de',                  'País Vasco',
                   'Rioja, La',                       'Ceuta',
                     'Melilla']
Length: 19, dtype: str

In [8]:
df_empr_const["tipo"].unique()

<StringArray>
[                           'Mercantiles',
                    'Sociedades anónimas',
 'Sociedades de responsabilidad limitada',
       'S. Comanditarias y S. Colectivas']
Length: 4, dtype: str

Tras revisar los tipos de empresa, nos damos cuenta de que "Mercantiles" es un total de las empresas SL, y SA, vamos a comprobarlo.

In [9]:
df_empr_const[df_empr_const["tipo"] == "Mercantiles"].sample(10)

,territorio,id_tiempo,tipo,numero_sociedades,capital
id_const,,,,,
2842,"Madrid, Comunidad de",201008,Mercantiles,990,58060000
417,Aragón,201002,Mercantiles,210,45750000
2692,"Madrid, Comunidad de",202302,Mercantiles,2389,76236000
2225,Extremadura,202503,Mercantiles,113,4483000
1718,Castilla - La Mancha,201203,Mercantiles,312,7501000
2110,Comunitat Valenciana,201605,Mercantiles,1102,25379000
817,"Balears, Illes",201308,Mercantiles,158,6135000
3257,"Navarra, Comunidad Foral de",201211,Mercantiles,72,41706000
103,Andalucía,201711,Mercantiles,1301,65057000


In [10]:
df_empr_const["tipo"].value_counts()

tipo
Mercantiles                               4199
Sociedades anónimas                       4199
Sociedades de responsabilidad limitada    4199
S. Comanditarias y S. Colectivas          4199
Name: count, dtype: int64

La razón de que los valores totales coinciden es porque hay una fila por fecha, independientemente de si hay constituidas

In [11]:
df_empr_const[df_empr_const["tipo"] == "Mercantiles"].shape

(4199, 5)

In [12]:
df_empr_const[(df_empr_const["territorio"] == "Melilla") & (df_empr_const["id_tiempo"] == 202505)].head(100)

,territorio,id_tiempo,tipo,numero_sociedades,capital
id_const,,,,,
3991,Melilla,202505,Mercantiles,10,1037000
8190,Melilla,202505,Sociedades anónimas,0,0
12389,Melilla,202505,Sociedades de responsabilidad limitada,10,1037000
16588,Melilla,202505,S. Comanditarias y S. Colectivas,0,0


In [13]:
df_empr_const[(df_empr_const["territorio"] == "Canarias") & (df_empr_const["id_tiempo"] == 202006)].head(100)

,territorio,id_tiempo,tipo,numero_sociedades,capital
id_const,,,,,
956,Canarias,202006,Mercantiles,186,155088000
5155,Canarias,202006,Sociedades anónimas,0,0
9354,Canarias,202006,Sociedades de responsabilidad limitada,186,155088000
13553,Canarias,202006,S. Comanditarias y S. Colectivas,0,0


In [14]:
df_empr_const[(df_empr_const["territorio"] == "Andalucía") & (df_empr_const["id_tiempo"] == 202407)].head(100)

,territorio,id_tiempo,tipo,numero_sociedades,capital
id_const,,,,,
23,Andalucía,202407,Mercantiles,1564,78298000
4222,Andalucía,202407,Sociedades anónimas,2,1840000
8421,Andalucía,202407,Sociedades de responsabilidad limitada,1562,76458000
12620,Andalucía,202407,S. Comanditarias y S. Colectivas,0,0


In [15]:
df_empr_const.columns

Index(['territorio', 'id_tiempo', 'tipo', 'numero_sociedades', 'capital'], dtype='str')

In [16]:
df_empr_const.duplicated().sum()

np.int64(0)

Confirmamos nuestras sospechas, procederemos en el paso de transformación a eliminar esas filas. df_empr_const["tipo"] == "Mercantiles"

In [17]:
df_empr_dis = pd.read_csv('../files/data_raw/empresas_disueltas.csv', index_col=0)

In [18]:
df_empr_dis.sample(5)

,territorio,id_tiempo,razon,numero_sociedades
id_dis,,,,
9979,Castilla - La Mancha,202308,Otras,1
7707,País Vasco,201005,Por fusión,12
9306,Canarias,202406,Otras,6
5776,Castilla - La Mancha,202312,Por fusión,0
6127,Cataluña,201302,Por fusión,15


In [19]:
df_empr_dis.info()

<class 'pandas.DataFrame'>
RangeIndex: 12597 entries, 1 to 12597
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   territorio         12597 non-null  str  
 1   id_tiempo          12597 non-null  int64
 2   razon              12597 non-null  str  
 3   numero_sociedades  12597 non-null  int64
dtypes: int64(2), str(2)
memory usage: 393.8 KB


In [20]:
df_empr_dis.shape

(12597, 4)

In [21]:
df_empr_dis.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_tiempo,12597.0,201677.914027,531.903084,200801.0,201208.0,201703.0,202110.0,202605.0
numero_sociedades,12597.0,32.498134,71.886292,0.0,2.0,8.0,31.0,1114.0


In [22]:
df_empr_dis.describe(include='string').T

,count,unique,top,freq
territorio,12597,19,Andalucía,663
razon,12597,3,Voluntaria,4199


In [23]:
df_empr_dis["razon"].value_counts()

razon
Voluntaria    4199
Por fusión    4199
Otras         4199
Name: count, dtype: int64

In [24]:
df_empr_dis.duplicated().sum()

np.int64(0)

La razón por la que los valores totales coinciden es porque hay una fila por fecha

-------------------------

In [25]:
df_ipc = pd.read_csv('../files/data_raw/ipc.csv')

In [26]:
df_ipc.sample(10)

,id_tiempo,id_territorio,id_sector,id_medida,valor_ipc
150566,200502,10,3,2,0.000
151644,201212,10,4,2,-0.400
227324,200510,14,12,4,3.800
88053,201310,6,6,1,95.520
153358,201608,10,5,4,-3.200
17150,201307,2,1,3,1.600
12502,201003,1,11,3,2.600
300144,201703,19,5,1,84.776
157871,200610,10,9,3,-8.900
137414,200205,9,6,1,77.532


In [27]:
df_ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 328162 entries, 0 to 328161
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id_tiempo      328162 non-null  int64  
 1   id_territorio  328162 non-null  int64  
 2   id_sector      328162 non-null  int64  
 3   id_medida      328162 non-null  int64  
 4   valor_ipc      328162 non-null  float64
dtypes: float64(1), int64(4)
memory usage: 12.5 MB


In [28]:
df_ipc.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_tiempo,328162.0,201377.778817,705.032930,200201.0,200802.0,201403.0,202004.0000,202606.000
id_territorio,328162.0,10.499942,5.766320,1.0,5.0,10.0,15.0000,20.000
id_sector,328162.0,7.499960,4.031155,1.0,4.0,7.0,11.0000,14.000
id_medida,328162.0,2.500000,1.118033,1.0,2.0,2.5,3.0000,4.000
valor_ipc,328162.0,21.500742,37.058520,-22.4,0.1,1.5,37.3585,258.216


In [29]:
df_ipc[df_ipc["valor_ipc"] == -22.4].head()

,id_tiempo,id_territorio,id_sector,id_medida,valor_ipc
185797,202308,12,5,3,-22.4


In [ ]:
df_ipc[df_ipc["valor_ipc"] < 0].sample(10)

,id_tiempo,id_territorio,id_sector,id_medida,valor_ipc
125379,200403,8,9,4,-2.4
90126,201112,6,7,4,-4.1
116758,201407,8,2,3,-2.3
289058,201303,18,9,3,-3.2
153080,201505,10,5,3,-1.8
114001,202407,7,14,2,-0.7
236619,201207,15,6,4,-1.3
259387,201909,16,12,2,-0.4
73425,201112,5,7,3,-2.8
307001,200705,19,10,4,-1.1


In [31]:
df_ipc.duplicated().sum()

np.int64(0)

----------------

In [32]:
df_sectores_ipc = pd.read_csv('../files/data_raw/sectores_ipc.csv')

In [33]:
df_sectores_ipc.sample(10)

,id_sector,nombre_sector
2,3,Bebidas alcohólicas y tabaco
12,13,Seguros y servicios financieros
11,12,Restaurantes y servicios de alojamiento
7,8,Transporte
3,4,Vestido y calzado
13,14,"Cuidado personal, protección social, y bienes ..."
4,5,"Vivienda, agua, electricidad, gas y otros comb..."
10,11,Enseñanza
6,7,Sanidad
5,6,"Muebles, artículos del hogar y artículos para ..."


In [34]:
df_sectores_ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_sector      14 non-null     int64
 1   nombre_sector  14 non-null     str  
dtypes: int64(1), str(1)
memory usage: 356.0 bytes


In [35]:
df_sectores_ipc.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_sector,14.0,7.5,4.1833,1.0,4.25,7.5,10.75,14.0


In [36]:
df_sectores_ipc.describe(include='string').T

,count,unique,top,freq
nombre_sector,14,14,Índice general,1


In [37]:
df_sectores_ipc["nombre_sector"].unique()

<StringArray>
[                                                                    'Índice general',
                                                 'Alimentos y bebidas no alcohólicas',
                                                       'Bebidas alcohólicas y tabaco',
                                                                  'Vestido y calzado',
                             'Vivienda, agua, electricidad, gas y otros combustibles',
 'Muebles, artículos del hogar y artículos para el mantenimiento corriente del hogar',
                                                                            'Sanidad',
                                                                         'Transporte',
                                                       'Información y comunicaciones',
                                         'Actividades recreativas, deporte y cultura',
                                                                          'Enseñanza',
                             

------------------

In [38]:
df_territorio = pd.read_csv('../files/data_raw/territorio.csv')

In [39]:
df_territorio.sample(10)

,id_territorio,nombre_territorio
5,6,Canarias
18,19,Ceuta
1,2,Andalucía
0,1,Nacional
14,15,"Murcia, Región de"
11,12,Extremadura
16,17,País Vasco
2,3,Aragón
17,18,"Rioja, La"
10,11,Comunitat Valenciana


In [40]:
df_territorio.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_territorio      20 non-null     int64
 1   nombre_territorio  20 non-null     str  
dtypes: int64(1), str(1)
memory usage: 452.0 bytes


In [41]:
df_territorio.describe(include='string').T

,count,unique,top,freq
nombre_territorio,20,20,Nacional,1


In [42]:
df_territorio["nombre_territorio"].unique()

<StringArray>
[                   'Nacional',                   'Andalucía',
                      'Aragón',     'Asturias, Principado de',
              'Balears, Illes',                    'Canarias',
                   'Cantabria',             'Castilla y León',
        'Castilla - La Mancha',                    'Cataluña',
        'Comunitat Valenciana',                 'Extremadura',
                     'Galicia',        'Madrid, Comunidad de',
           'Murcia, Región de', 'Navarra, Comunidad Foral de',
                  'País Vasco',                   'Rioja, La',
                       'Ceuta',                     'Melilla']
Length: 20, dtype: str

-----------------

In [43]:
df_tiempo = pd.read_csv('../files/data_raw/tiempo.csv')

In [44]:
df_tiempo.sample(10)

,id_tiempo,anio,mes,nombre_mes
101,201712,2017,12,Diciembre
20,202409,2024,9,Septiembre
233,200612,2006,12,Diciembre
4,202601,2026,1,Enero
200,200909,2009,9,Septiembre
183,201102,2011,2,Febrero
246,200511,2005,11,Noviembre
129,201508,2015,8,Agosto
185,201012,2010,12,Diciembre
160,201301,2013,1,Enero


In [45]:
df_tiempo.info()

<class 'pandas.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id_tiempo   294 non-null    int64
 1   anio        294 non-null    int64
 2   mes         294 non-null    int64
 3   nombre_mes  294 non-null    str  
dtypes: int64(3), str(1)
memory usage: 9.3 KB


In [46]:
df_tiempo.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
id_tiempo,294.0,201381.948980,708.657083,200201.0,200802.25,201403.5,202004.75,202606.0
anio,294.0,2013.755102,7.087548,2002.0,2008.00,2014.0,2020.00,2026.0
mes,294.0,6.438776,3.457394,1.0,3.00,6.0,9.00,12.0


In [47]:
df_tiempo.duplicated().sum()

np.int64(0)

---------------------

In [48]:
df_tipo_medida = pd.read_csv('../files/data_raw/tipo_medida.csv')

In [49]:
df_tipo_medida.head(10)

,id_medida,nombre_medida
0,1,Índice
1,2,Variación mensual
2,3,Variación anual
3,4,Variación en lo que va de año


In [50]:
df_tipo_medida.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_medida      4 non-null      int64
 1   nombre_medida  4 non-null      str  
dtypes: int64(1), str(1)
memory usage: 196.0 bytes


In [51]:
df_tipo_medida.value_counts()

id_medida  nombre_medida                
1          Índice                           1
2          Variación mensual                1
3          Variación anual                  1
4          Variación en lo que va de año    1
Name: count, dtype: int64